# Phase 3 — DoRA Fine-Tuning — intent-classifier

This notebook trains and evaluates DoRA adapters for all 5 models × 4 configs on Google Colab.

DoRA (Weight-Decomposed Low-Rank Adaptation) decomposes each weight matrix into
**magnitude** and **direction** components, then applies LoRA only to the direction.
It reuses the same A/B/C/D configs as LoRA — the only implementation difference is
`use_dora=True` in `LoraConfig`.

**What runs here:**
1. Clone the repo from GitHub
2. Install dependencies
3. HF authentication (token stored in Colab Secrets)
4. GPU environment check
5. Configuration
6. Prepare data splits
7. Smoke test — 10 steps to validate the pipeline
8. Full training — all (model × config) combinations
9. Validation — load each adapter, evaluate on val split
10. Test evaluation (locked — run deliberately)
11. Inference sanity check
12. Download reports

**Note:** Plot generation is done offline. Reports are JSON files — download them,
then run `plot_dora_results.py` locally.

## 1. Clone repository

In [1]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

print(f"Repo at: {REPO_DIR}")

Repo at: /content/intent-classifier


## 2. Install dependencies

In [2]:
%pip install -q \
    torch \
    "torchao>=0.16.0" \
    transformers \
    accelerate \
    "peft>=0.14.0" \
    trl \
    datasets \
    bitsandbytes \
    huggingface_hub \
    python-dotenv \
    sentencepiece \
    protobuf

print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 49.1 MB/s eta 0:00:00
Dependencies installed.


## 3. Hugging Face authentication

Store your `HF_TOKEN` in **Colab Secrets** (key icon in the left sidebar).
Required for:
- Downloading `meta-llama/Llama-3.2-1B-Instruct` (gated model)
- Uploading trained adapters to `kon172verma/intent-classifier`

In [3]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("WARNING: HF_TOKEN secret is empty.")
except Exception as e:
    print(f"Not running in Colab or secret missing: {e}")

HF_TOKEN loaded from Colab Secrets.


## 4. GPU environment check

In [4]:
import subprocess
import platform
import torch

print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("GPU     : not available — training will be very slow on CPU")

# Verify DoRA is supported in installed PEFT version
import peft
print(f"PEFT    : {peft.__version__}  (DoRA requires >= 0.9.0)")

Python  : 3.13.15
PyTorch : 2.11.0+cu128
CUDA    : 12.8
GPU     : NVIDIA L4  (23.7 GB VRAM)


PEFT    : 0.20.0  (DoRA requires >= 0.9.0)


## 5. Configuration

Edit the variables below to control the experiment matrix.

| Variable | Description |
|---|---|
| `MODELS_TO_RUN` | Which models to train (subset or all 5) |
| `CONFIGS_TO_RUN` | Which configs (A / B / C / D) |
| `DATASET_SIZE` | `"1k"` or `"10k"` |
| `DEVICE` | `"auto"` (recommended) or `"cuda"` |

**Config legend (same as LoRA — DoRA adds `use_dora=True`):**
| Config | Description |
|---|---|
| A | Light — Q+V only, r=8 |
| B | Standard — full attention, r=16 |
| C | Wide — attn+MLP, r=16 |
| D | Heavy — attn+MLP, r=32 |

In [5]:
import sys

REPO_DIR  = "/content/intent-classifier"
SRC_DIR   = f"{REPO_DIR}/finetune_DoRA/src"
DATA_DIR  = f"{REPO_DIR}/finetune_DoRA/data"

# ── Experiment matrix ──────────────────────────────────────────────────────

ALL_CONFIGS = ["A", "B", "C", "D"]

MODELS_TO_RUN  = ALL_MODELS   # change to a subset for partial runs
CONFIGS_TO_RUN = ALL_CONFIGS  # change to e.g. ["A", "B"] for partial

DATASET_SIZE = "1k"           # "1k" or "10k"
DEVICE       = "auto"

SPLIT_DIR = f"{DATA_DIR}/{DATASET_SIZE}"

# Add repo root to path so finetune_lib is importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Configuration:")
print(f"  Models   : {MODELS_TO_RUN}")
print(f"  Configs  : {CONFIGS_TO_RUN}")
print(f"  Dataset  : {DATASET_SIZE}")
print(f"  Device   : {DEVICE}")

Configuration:
  Models   : ['smollm2-360m', 'qwen2.5-0.5b', 'qwen3-0.6b', 'llama3.2-1b', 'smollm2-1.7b']
  Configs  : ['A', 'B', 'C', 'D']
  Dataset  : 1k
  Device   : auto


## 6. Prepare data splits

In [6]:
import os
import subprocess
import sys

os.makedirs(SPLIT_DIR, exist_ok=True)

PREP_SCRIPT = f"{REPO_DIR}/finetune_DoRA/src/prepare_dora_data.py"
cmd = [
    sys.executable, "-u", PREP_SCRIPT,
    "--dataset-size", DATASET_SIZE,
    "--out-dir",      DATA_DIR,
]
print("Preparing data splits...")
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
if proc.returncode != 0:
    raise RuntimeError("prepare_dora_data.py failed")
print("Data preparation complete.")

Preparing data splits...

  Dataset size : 1k
  Source dir   : /content/intent-classifier/dataset_full
  Output dir   : /content/intent-classifier/finetune_DoRA/data/1k

  Wrote   800 examples  →  /content/intent-classifier/finetune_DoRA/data/1k/train.jsonl
  Wrote   100 examples  →  /content/intent-classifier/finetune_DoRA/data/1k/val.jsonl
  Wrote   100 examples  →  /content/intent-classifier/finetune_DoRA/data/1k/test.jsonl
  Wrote   100 examples  →  /content/intent-classifier/finetune_DoRA/data/1k/test_anchor.jsonl

  Split summary (1k):
    train        : 800
    val          : 100
    test         : 100
    test_anchor  : 100  (sample_0001 only)

  Done → /content/intent-classifier/finetune_DoRA/data/1k
Data preparation complete.


## 7. Smoke test

Runs **10 training steps** on the first model + config A to validate the
complete pipeline (data loading → tokenisation → PEFT DoRA → training → report save)
without committing to a full run.

**Safe to skip** if you have already validated the pipeline.

In [7]:
import subprocess
import sys
import time

SMOKE_MODEL  = MODELS_TO_RUN[0]
SMOKE_CONFIG = "A"

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/dora_train.py",
    "--model",        SMOKE_MODEL,
    "--lora-config",  SMOKE_CONFIG,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
    "--smoke-test",
    "--no-push",
]
print("Running smoke test:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
elapsed = time.time() - t0
print(f"\nSmoke test {'PASSED' if proc.returncode == 0 else 'FAILED'} in {elapsed:.0f}s")

Running smoke test: /content/intent-classifier/finetune_DoRA/src/dora_train.py --model smollm2-360m --lora-config A --dataset-size 1k --device auto --smoke-test --no-push


  DoRA Training — smollm2-360m_A_1k
  Model        : HuggingFaceTB/SmolLM2-360M-Instruct
  LoRA config  : A — Light — attention Q/V only, rank 8
  Dataset      : 1k
  Device       : cuda
  Adapter dest : /content/intent-classifier/finetune_DoRA/adapters/smollm2-360m_A_1k
  HF repo      : kon172verma/intent-classifier-experiments/v2.0/smollm2-360m_DoRA_A_1k_<timestamp>
  Mode         : SMOKE TEST (10 steps only)

  Train : 800 examples
  Val   : 100 examples

  Loading tokenizer: HuggingFaceTB/SmolLM2-360M-Instruct
  Loading model:     HuggingFaceTB/SmolLM2-360M-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1575.18it/s]

  Trainable params : 860,160  (0.237%)
  Total params     : 362,681,280
  Grad checkpoint  : enabled

  Tokenizing

## 8. Train

Runs all (model × config) combinations via `run_dora_experiments.py`.

After each run the adapter is:
1. Saved locally → `finetune_DoRA/adapters/{run_tag}/`
2. Pushed to HF  → `kon172verma/intent-classifier/DoRA/{run_tag}/`

Training reports (JSON with full `log_history`) are saved to
`finetune_DoRA/reports_training/`.

**Estimated runtimes on L4 GPU (1k dataset, DoRA ~10-20% slower than LoRA):**

| Model | Config A | Config B | Config C | Config D |
|---|---|---|---|---|
| smollm2-360m   | ~9 min  | ~11 min | ~14 min | ~17 min |
| qwen2.5-0.5b   | ~11 min | ~15 min | ~17 min | ~20 min |
| qwen3-0.6b     | ~11 min | ~15 min | ~17 min | ~20 min |
| llama3.2-1b    | ~22 min | ~28 min | ~32 min | ~38 min |
| smollm2-1.7b   | ~28 min | ~34 min | ~39 min | ~47 min |

In [8]:
import subprocess
import sys
import time

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/run_dora_experiments.py",
    "--models",       *MODELS_TO_RUN,
    "--configs",      *CONFIGS_TO_RUN,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
    "--gradient-checkpointing",
]
print("Running:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
elapsed = time.time() - t0
print(f"\nTraining {'complete' if proc.returncode == 0 else 'FAILED'} in {elapsed/60:.1f} min")

Streaming output truncated to the last 5000 lines.

 85%|████████▍ | 11/13 [00:04<00:00,  2.38it/s]

 92%|█████████▏| 12/13 [00:04<00:00,  2.52it/s]

100%|██████████| 13/13 [00:05<00:00,  3.01it/s]
                                                 

                                               
{'eval_loss': '0.06685', 'eval_model_preparation_time': '0.0139', 'eval_runtime': '5.375', 'eval_samples_per_second': '18.61', 'eval_steps_per_second': '2.419', 'eval_entropy': '0.003894', 'eval_num_tokens': '9.14e+05', 'eval_mean_token_accuracy': '0.9904', 'epoch': '3'}

 75%|███████▌  | 150/200 [10:29<03:07,  3.75s/it]

100%|██████████| 13/13 [00:05<00:00,  3.01it/s]
  [Accuracy] step=150  train=1.0000  val=0.9700


                                               
100%|██████████| 200/200 [14:05<00:00,  3.78s/it]
                                                 
{'loss': '0.0001305', 'grad_norm': '0.004748', 'learning_rate': '6.835e-09', 'entropy': '0.0009663', 'num_tokens': '1.219e+06', 'mean

## 9. Validation

Evaluates each adapter on the validation split.
Reports are saved to `finetune_DoRA/reports_validation/`.

In [9]:
import subprocess
import sys
import time

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/run_dora_experiments.py",
    "--models",       *MODELS_TO_RUN,
    "--configs",      *CONFIGS_TO_RUN,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
    "--skip-training",  # adapters already uploaded; load from HF
]
print("Running validation:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
print(f"\nValidation {'complete' if proc.returncode == 0 else 'FAILED'} in {time.time()-t0:.0f}s")

Running validation: /content/intent-classifier/finetune_DoRA/src/run_dora_experiments.py --models smollm2-360m qwen2.5-0.5b qwen3-0.6b llama3.2-1b smollm2-1.7b --configs A B C D --dataset-size 1k --device auto --skip-training


  DoRA Experiment Runner
  Models   : ['smollm2-360m', 'qwen2.5-0.5b', 'qwen3-0.6b', 'llama3.2-1b', 'smollm2-1.7b']
  Configs  : ['A', 'B', 'C', 'D']
  Dataset  : 1k
  Device   : auto
  Runs     : 20 training + 20 val eval
  Training : SKIPPED

────────────────────────────────────────────────────────────
  VAL    [1/20] smollm2-360m_A_1k
────────────────────────────────────────────────────────────

  DoRA Evaluation — smollm2-360m_A_1k
  Split    : val  (100 examples)
  Source   : kon172verma/intent-classifier-experiments/v2.0/smollm2-360m_DoRA_A_1k_20260826-163545
  Device   : cuda

  Loading base model: HuggingFaceTB/SmolLM2-360M-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1

## 10. Test evaluation (locked)

Run this cell deliberately **once** after reviewing validation results.

Reports saved to `finetune_DoRA/reports_test/`.

In [10]:
import subprocess
import sys
import time

for model_key in MODELS_TO_RUN:
    for cfg in CONFIGS_TO_RUN:
        run_tag = f"{model_key}_{cfg}_{DATASET_SIZE}"
        cmd = [
            sys.executable, "-u", f"{SRC_DIR}/dora_validate.py",
            "--model",        model_key,
            "--lora-config",  cfg,
            "--dataset-size", DATASET_SIZE,
            "--split",        "test",
            "--device",       DEVICE,
        ]
        print(f"\n  Testing {run_tag}...")
        t0 = time.time()
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=REPO_DIR,
        )
        _buf: list[str] = []
        while True:
            ch = proc.stdout.read(1)
            if not ch:
                if _buf:
                    line = "".join(_buf)
                    if not line.startswith("Failed to load "):
                        print(line, end="", flush=True)
                break
            _buf.append(ch)
            if ch in ("\r", "\n"):
                line = "".join(_buf)
                if not line.startswith("Failed to load "):
                    print(line, end="", flush=True)
                _buf = []
        proc.stdout.close()
        proc.wait()
        print(f"  {'OK' if proc.returncode == 0 else 'FAILED'} in {time.time()-t0:.0f}s")


  Testing smollm2-360m_A_1k...

  DoRA Evaluation — smollm2-360m_A_1k
  Split    : test  (100 examples)
  Source   : kon172verma/intent-classifier-experiments/v2.0/smollm2-360m_DoRA_A_1k_20260826-163545
  Device   : cuda

  Loading base model: HuggingFaceTB/SmolLM2-360M-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1449.07it/s]
  Loading adapter from HF: kon172verma/intent-classifier-experiments/v2.0/smollm2-360m_DoRA_A_1k_20260826-163545
  [  20/100]  running_acc=0.300
  [  40/100]  running_acc=0.325
  [  60/100]  running_acc=0.417
  [  80/100]  running_acc=0.450
  [ 100/100]  running_acc=0.460
  Report pushed  : kon172verma/intent-classifier-experiments/reports/v2.0/dora/reports_test/smollm2-360m_A_1k_test_20260826_235909.json

  Accuracy   : 0.4600  (46/100)
  Avg latency: 149.3 ms
  Peak memory: 768 MB
  Report     : /content/intent-classifier/finetune_DoRA/reports_test/smollm2-360m_A_1k_test_2026

## 11. Inference sanity check

Manually inspect model outputs on a few hand-picked examples.
This catches output format issues that aggregate accuracy metrics would not surface.

In [11]:
import json
import sys
import torch
from pathlib import Path

# Pick any run_tag to inspect
INSPECT_MODEL  = MODELS_TO_RUN[0]
INSPECT_CONFIG = CONFIGS_TO_RUN[0]

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from finetune_lib import (
    CURRENT_VERSION, FINETUNE_MODEL_REGISTRY, HF_EXPERIMENTS_REPO,
    answer_to_tool_id, apply_chat_template_safe, build_chat_messages,
    extract_prediction, load_jsonl, tool_id_to_answer,
)
from finetune_lib.registry import find_latest_experiment
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = FINETUNE_MODEL_REGISTRY[INSPECT_MODEL]
run_tag  = f"{INSPECT_MODEL}_{INSPECT_CONFIG}_{DATASET_SIZE}"
local_adapter = Path(REPO_DIR) / "finetune_DoRA" / "adapters" / run_tag

entry = find_latest_experiment(
    technique="DoRA",
    model_key=INSPECT_MODEL,
    lora_config=INSPECT_CONFIG,
    dataset_size=DATASET_SIZE,
    version=CURRENT_VERSION,
)
hf_sub = entry["hf_subfolder"] if entry else None
if not local_adapter.exists() and not hf_sub:
    raise RuntimeError(
        f"No adapter found for {run_tag}. Smoke tests do not save/push "
        "adapters; run full training first, or set INSPECT_MODEL / "
        "INSPECT_CONFIG to a completed run."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if device.type == "cuda" else torch.float32

tokenizer  = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map={"": device})
if local_adapter.exists():
    print(f"Loading local adapter: {local_adapter}")
    model = PeftModel.from_pretrained(base_model, str(local_adapter))
elif hf_sub:
    print(f"Loading HF adapter: {HF_EXPERIMENTS_REPO}/{hf_sub}")
    model = PeftModel.from_pretrained(base_model, HF_EXPERIMENTS_REPO, subfolder=hf_sub)
model.eval()

val_file = Path(f"{DATA_DIR}/{DATASET_SIZE}/val.jsonl")
examples = load_jsonl(val_file)[:5]  # first 5 examples

for i, ex in enumerate(examples, 1):
    msgs = build_chat_messages(ex, include_answer=False)
    text = apply_chat_template_safe(tokenizer, msgs, INSPECT_MODEL, add_generation_prompt=True)
    inp  = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=8, do_sample=False,
                             pad_token_id=int(tokenizer.eos_token_id))
    new_ids = out[0][inp["input_ids"].shape[1]:]
    raw     = tokenizer.decode(new_ids, skip_special_tokens=True)
    pred    = extract_prediction(raw)
    answer_id = answer_to_tool_id(ex)
    pred_tool = tool_id_to_answer(ex, pred)
    correct = pred == answer_id
    print(f"[{i}] user_request={ex['user_request'][:60]!r}")
    print(f"     expected={answer_id!r} ({ex['answer']})  got={pred!r} ({pred_tool})  raw={raw!r}  {'✓' if correct else '✗'}")
    print()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading local adapter: /content/intent-classifier/finetune_DoRA/adapters/smollm2-360m_A_1k
[1] user_request="I'm locked out of my car, can someone come help?"
     expected='n' (roadside_assistance)  got='p' (nav_route_planner)  raw='p'  ✗

[2] user_request='Do I have a contact called Sarah in my list?'
     expected='c' (corp-contact-sync)  got='c' (corp-contact-sync)  raw='c'  ✓

[3] user_request='What is the population of Japan?'
     expected='-' (none)  got='-' (none)  raw='-'  ✓

[4] user_request='Schedule a meeting with Tanaka-san at 3 pm tomorrow.'
     expected='o' (corp-scheduler-app)  got='n' (in_car_analytics)  raw='n'  ✗

[5] user_request="Send a text to mom saying I'll be late."
     expected='d' (sms_messenger)  got='e' (news_briefing)  raw='e'  ✗



## 12. Download reports

Zips and downloads all JSON report files so you can run plot scripts locally.

Contents of the zip:
- `reports_training/` — training curves + log history (one JSON per run)
- `reports_validation/` — validation accuracy (one JSON per run)
- `reports_test/` — test accuracy + per-tool metrics (one JSON per run)

In [12]:
import os
import shutil
from pathlib import Path

try:
    from google.colab import files  # type: ignore

    dora_dir = Path(REPO_DIR) / "finetune_DoRA"
    zip_base = "/content/dora_reports"

    report_dirs = {
        "reports_training":   dora_dir / "reports_training",
        "reports_validation": dora_dir / "reports_validation",
        "reports_test":       dora_dir / "reports_test",
    }

    staging = Path("/content/_dora_reports_staging")
    staging.mkdir(exist_ok=True)
    for name, src in report_dirs.items():
        if src.exists() and any(src.iterdir()):
            shutil.copytree(src, staging / name, dirs_exist_ok=True)
        else:
            print(f"  Skipping {name} (empty or missing)")

    shutil.make_archive(zip_base, "zip", staging)
    print(f"Created {zip_base}.zip")
    files.download(f"{zip_base}.zip")

except ImportError:
    print("Not running in Colab — copy reports manually from:")
    for name in ["reports_training", "reports_validation", "reports_test"]:
        print(f"  {REPO_DIR}/finetune_DoRA/{name}/")

Created /content/dora_reports.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>